# Free Embeddings Intuition


[Step 7 - Free embeddings]

> **MLCourse - Agentic AI - Embeddings**
> Stage in the capstone: EMBED - every chunk becomes a vector; query becomes one too.

# What you will learn

1. What an embedding is: text mapped to a vector whose geometry encodes meaning.
2. How to embed sentences locally with a FREE model (all-MiniLM-L6-v2, 384 dims).
3. How to compute cosine similarity BY HAND with numpy and read it as a heatmap.
4. How batch embedding (`embed_documents`) differs from query embedding (`embed_query`).
5. How to swap in Ollama's nomic-embed-text (768 dims) when it is available.

Everything in this notebook runs 100% on your machine: no API keys, no cloud
calls after the one-time model download. That makes it safe to rerun freely
while you build intuition.

In [1]:
# ---------------------------------------------------------------------------
# Setup cell (identical in every MLCourse notebook): imports, TRACK walker,
# DATA folder creation, .env loading, matplotlib inline magic - guarded so
# the file also runs as a plain script outside Jupyter.
# ---------------------------------------------------------------------------
from pathlib import Path


def find_track(start: Path, target: str = "03_agentic_ai") -> Path:
    """Climb parent folders until a directory named ``target`` shows up."""
    for candidate in [start, *start.parents]:
        probe = candidate / target
        if probe.is_dir():
            return probe
    raise FileNotFoundError(
        f"Could not find '{target}' above {start}. "
        "Open this notebook from inside the MLCourse repository."
    )


TRACK = find_track(Path.cwd())       # .../MLCourse/03_agentic_ai
DATA = TRACK / "data"                # one shared data folder for the track
DATA.mkdir(exist_ok=True)            # no-op when it already exists

from dotenv import load_dotenv       # noqa: E402  reads KEY=value files

load_dotenv()                        # .env beside the current directory
load_dotenv(TRACK / ".env")          # .env at the track root

try:
    # Jupyter/IPython defines get_ipython(); plain python scripts do not,
    # hence the guard instead of a raw %matplotlib line.
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                             # silently skip the magic outside IPython

import matplotlib.pyplot as plt      # noqa: E402

print("[setup] TRACK:", TRACK)
print("[setup] DATA :", DATA)

[setup] TRACK: D:\projects\python\MLCourse\03_agentic_ai
[setup] DATA : D:\projects\python\MLCourse\03_agentic_ai\data


### Why embeddings matter for agents

Keyword search fails on paraphrases: "rabbit hole" does not literally match
"fell into a deep hole". Embeddings bridge that vocabulary gap because they
place texts by MEANING, not by shared words. The capstone uses them twice -
once to index every document chunk, once per user question - so building
intuition for the geometry now pays off for every later module.

### Load the FREE local embedder and embed three sentences one at a time.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings   # official keyless wrapper
import numpy as np                                        # vector math toolbox

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"     # tiny + fast + 384 dims

# First call downloads ~90 MB into the local HuggingFace cache, then it is free.
embedder = HuggingFaceEmbeddings(model_name=MODEL_NAME)

SENTENCES = [
    "Alice follows the white rabbit down the rabbit hole.",   # topic A
    "Alice chased the rabbit and fell into a deep hole.",     # topic A, paraphrased
    "Quantum processors factor large prime numbers fast.",    # topic B, unrelated
]

# embed_query() is the QUERY-TIME method: exactly one string in, one vector out.
vec_a = np.array(embedder.embed_query(SENTENCES[0]))
vec_b = np.array(embedder.embed_query(SENTENCES[1]))
vec_c = np.array(embedder.embed_query(SENTENCES[2]))

print("single vector length :", len(vec_a), "(MiniLM dimensionality = 384)")
print("first 5 floats       :", np.round(vec_a[:5], 4))
print("vector norm          :", round(float(np.linalg.norm(vec_a)), 4))


### Similarity math, kept simple

Two formulas cover 95% of embedding work:

- dot(a, b) = sum of elementwise products (cheap, but rewards big magnitudes)
- cosine(a, b) = dot(a, b) / (norm(a) * norm(b)) -> angle between directions only

Cosine ignores how LONG each vector is, which suits text: a longer sentence
should not rank higher just for having more words. Handy identity: once both
vectors are normalized to unit length, dot == cosine - we exploit that below
by dividing once and then using a single matrix multiply for all pairs.

### Build the full pairwise cosine similarity matrix manually, then visualize it.


In [ ]:
import pandas as pd

vectors = np.vstack([vec_a, vec_b, vec_c])               # shape (3, 384)


def cosine_similarity_matrix(vecs: np.ndarray) -> np.ndarray:
    """Pairwise cosine similarity for a stack of vectors.

    cos(a, b) = dot(a, b) / (norm(a) * norm(b))
              = angle between the two vectors, ignoring their lengths.
    Row i vs column j holds cos(vector_i, vector_j); diagonal is always 1.0.
    """
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)  # each row's length
    unit = vecs / norms                                  # project onto unit sphere
    return unit @ unit.T                                 # dot of unit vecs = cosine


sim = cosine_similarity_matrix(vectors)

LABELS = ["rabbit hole", "chased rabbit", "quantum primes"]
sim_df = pd.DataFrame(np.round(sim, 3), index=LABELS, columns=LABELS)
print(sim_df)                                            # ASCII-friendly table

# Same numbers as a heatmap: bright diagonal, bright A-A pair, dim cross-pair.
fig, ax = plt.subplots(figsize=(4.8, 3.8))
try:
    import seaborn as sns                                # pretty stats plotting
    sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="viridis",
                vmin=0.0, vmax=1.0, square=True, ax=ax,
                cbar_kws={"label": "cosine"})
except ImportError:                                      # seaborn is optional
    im = ax.imshow(sim, cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(3), LABELS, rotation=45, ha="right")
    ax.set_yticks(range(3), LABELS)
    fig.colorbar(im, ax=ax, label="cosine")
ax.set_title("MiniLM cosine similarity")
plt.tight_layout()
plt.show()

print("\nRead it: related sentences score ~%.2f, unrelated pair ~%.2f."
      % (sim[0, 1], sim[0, 2]))
print("High similarity on paraphrases, LOW across topics = geometry works.")


### Batch embedding with embed_documents

embed_query answers "how do I embed ONE question?" - but indexing a corpus
needs thousands of chunks embedded efficiently. That is embed_documents' job:
pass a list of strings, get one vector per string back, computed as a batch.
At index time module 08 will feed entire chunk lists through this method.

### INDEX-TIME twin of embed_query: whole corpus in, matrix of vectors out.


In [ ]:
batch_texts = SENTENCES + SENTENCES                      # doubled to fake a batch
batch_vecs = np.array(embedder.embed_documents(batch_texts))

print("batch shape           :", batch_vecs.shape, "= (num_texts, dims)")
same_text_same_vector = bool(np.allclose(batch_vecs[0], batch_vecs[3]))
print("row 0 == row 3        :", same_text_same_vector,
      "(identical text -> identical vector)")

# Cross-check: batch output must agree with the single-query vectors above.
matches_vec_a = bool(np.allclose(batch_vecs[0], vec_a))
print("row 0 matches vec_a   :", matches_vec_a,
      "(one model, consistent geometry)")


### Optional alternative: Ollama's nomic-embed-text (768 dims)

Another fully keyless option runs through a LOCAL Ollama server. It needs two
things this notebook cannot assume: Ollama installed and the model pulled
(`ollama pull nomic-embed-text`). The whole block is wrapped in try/except so
the notebook degrades gracefully - you see the same analysis either way.

### Same three sentences, same math - different (bigger, 768-dim) model.


In [ ]:
try:
    from langchain_ollama import OllamaEmbeddings        # official Ollama wrapper

    ollama_embedder = OllamaEmbeddings(model="nomic-embed-text")

    o_vecs = np.array(ollama_embedder.embed_documents(SENTENCES))
    o_sim = cosine_similarity_matrix(o_vecs)

    o_df = pd.DataFrame(np.round(o_sim, 3), index=LABELS, columns=LABELS)
    print(o_df)
    print("\nnomic-embed-text dimensionality:", o_vecs.shape[1],
          "(vs MiniLM's 384 - more room for nuance, more memory per vector)")
except Exception as exc:                 # server down, model not pulled, etc.
    print("[demo skipped] ollama pull nomic-embed-text")
    print("[demo skipped] reason:", type(exc).__name__, "-", exc)


### Takeaway

- Embeddings turn TEXT into GEOMETRY: closeness equals relatedness.
- all-MiniLM-L6-v2 gives useful 384-dim vectors offline and for free;
  nomic-embed-text upgrades you to 768 dims if Ollama is running.
- Cosine similarity (the normalized dot product) is THE number to read.
- embed_documents indexes the corpus; embed_query embeds the question -
  and BOTH sides must come from the SAME model forever after.

### Summary

We embedded three sentences with a free local model, hand-computed their
cosine similarity matrix with numpy, visualized it as a heatmap, proved
embed_documents batching is consistent with embed_query, and optionally
reproduced the whole flow with Ollama's 768-dim nomic-embed-text. Next: the
paid alternative, text-embedding-3-small, and what money actually buys.